In [1]:
from datasets import load_dataset

pubmed = load_dataset("pubmed_qa", "pqa_labeled", split="train")

/Users/suhaibbasir/Documents/CS/MSc/Thesis/.conda/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
len(pubmed)

1000

In [8]:
pubmed[0]

{'pubid': 21645374,
 'question': 'Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?',
 'context': {'contexts': ['Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has been less studied during PCD in plants.',
   'The following paper elucidates the role of mitochondrial dynamics during developmentally regulated PCD in vivo in A. madagascariensis. A single areole within a window stage leaf (PCD is occurring) was divided into three areas based on the progression of PCD; cells that will not undergo PCD (NPCD), ce

In [234]:
limit = 384

def chunker(contexts: list):
    chunks = []
    all_contexts = ' '.join(contexts).split('.')
    chunk = []
    for context in all_contexts:
        chunk.append(context)
        if len(chunk) >= 3 and len('.'.join(chunk)) > limit:
            # surpassed limit so add to chunks and reset
            chunks.append('.'.join(chunk).strip()+'.')
            # add some overlap between passages
            chunk = chunk[-2:]
    # if we finish and still have a chunk, add it
    if chunk is not None:
        chunks.append('.'.join(chunk))
    return chunks

chunks = chunker(pubmed[0]['context']['contexts'])
chunks

['Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature.',
 'The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has been less studied during PCD in plants. The following paper elucidates the role of mitochondrial dynamics during developmentally regulated PCD in vivo in A.',
 'The role of mitochondria during PCD has been recognized in animals; however, it has been less

In [235]:
ids = []
for i in range(len(chunks)):
    ids.append(f"{pubmed[0]['pubid']}-{i}")
ids


['21645374-0',
 '21645374-1',
 '21645374-2',
 '21645374-3',
 '21645374-4',
 '21645374-5',
 '21645374-6']

In [236]:
data = []
for record in pubmed:
    chunks = chunker(record['context']['contexts'])
    for i, context in enumerate(chunks):
        data.append({
            'id': f"{record['pubid']}-{i}",
            'context': context
        })

data[:10]

[{'id': '21645374-0',
  'context': 'Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature.'},
 {'id': '21645374-1',
  'context': 'The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has been less studied during PCD in plants. The following paper elucidates the role of mitochondrial dynamics during developmentally regulated PCD in vivo in A.'},
 {'id': '21645374-2',
  '

In [237]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForMaskedLM

tokenizer = AutoTokenizer.from_pretrained("naver/splade-cocondenser-ensembledistil")
model = AutoModelForMaskedLM.from_pretrained("naver/splade-cocondenser-ensembledistil")

In [238]:
data[0]['context']

'Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature.'

In [239]:
tokens = tokenizer(data[0]['context'], return_tensors="pt")
output = model(**tokens)

In [373]:
import torch 

def get_max_logits(output, tokens):
    return torch.max(
        torch.log(
            1 + torch.relu(output.logits)
        ) * tokens.attention_mask.unsqueeze(-1),
        dim=1)[0].squeeze().detach().cpu().numpy()

In [356]:
sparse_emb = get_max_logits(output, tokens)
sparse_emb.shape

[0. 0. 0. ... 0. 0. 0.]


(30522,)

In [358]:
sparse_emb

array([0., 0., 0., ..., 0., 0., 0.], dtype=float32)

In [251]:
indices = sparse_emb.nonzero().squeeze().cpu().tolist()
print(len(indices))

weights = sparse_emb[indices].cpu().tolist()

sparse_dict = dict(zip(indices, weights))
sparse_dict

174


{1000: 0.6246441006660461,
 1039: 0.45678871870040894,
 1052: 0.3088981807231903,
 1997: 0.15812599658966064,
 1999: 0.0719473734498024,
 2003: 0.6496514678001404,
 2024: 0.9411969184875488,
 2049: 0.31615185737609863,
 2083: 0.7597626447677612,
 2094: 1.950169563293457,
 2173: 0.3237403333187103,
 2239: 0.3950248062610626,
 2278: 0.2353711724281311,
 2290: 0.2457142472267151,
 2306: 0.4253380596637726,
 2331: 1.9602458477020264,
 2415: 0.6289482712745667,
 2427: 0.4244135618209839,
 2523: 0.018047992140054703,
 2537: 0.19568732380867004,
 2550: 0.6684800386428833,
 2565: 0.8162306547164917,
 2566: 1.0954251289367676,
 2597: 0.1979701966047287,
 2644: 0.2276630848646164,
 2754: 0.013309326022863388,
 2757: 0.9048289060592651,
 2832: 0.6024842858314514,
 2974: 0.6100065112113953,
 3030: 0.039797987788915634,
 3081: 0.12952162325382233,
 3102: 0.02347405068576336,
 3252: 0.3975687325000763,
 3269: 1.2144665718078613,
 3274: 0.7056938409805298,
 3280: 1.5106241703033447,
 3370: 0.53328686

In [252]:
values = sparse_emb[indices].cpu().tolist()
sparse = {'indices': indices, 'values': values}

In [253]:
idx2token = {idx: token for token, idx in tokenizer.get_vocab().items()}

In [267]:
sparse_dict_tokens = {
    idx2token[idx]: round(weight, 2) for idx, weight in zip(indices, values)
}
# sort so we can see most relevant tokens first
sparse_dict_tokens = {
    k: v for k, v in sorted(
        sparse_dict_tokens.items(),
        key=lambda item: item[1],
        reverse=True
    )
}
print(len(sparse_dict_tokens))
sparse_dict_tokens

174


{'pc': 3.02,
 'lace': 2.95,
 'programmed': 2.36,
 '##for': 2.28,
 'madagascar': 2.26,
 'death': 1.96,
 '##d': 1.95,
 'lattice': 1.81,
 'cell': 1.69,
 '##iensis': 1.64,
 'malaga': 1.6,
 '##get': 1.56,
 'regulated': 1.53,
 'die': 1.51,
 'lacey': 1.5,
 '##ono': 1.46,
 '##ole': 1.45,
 '##oles': 1.45,
 'transverse': 1.39,
 '##scu': 1.39,
 'leaves': 1.34,
 'cells': 1.31,
 'longitudinal': 1.31,
 'plant': 1.21,
 'plants': 1.16,
 'leaf': 1.15,
 'ap': 1.14,
 'organism': 1.12,
 'per': 1.1,
 'regulation': 1.03,
 'veins': 1.02,
 '##work': 1.0,
 'organisms': 1.0,
 'are': 0.94,
 'modified': 0.93,
 'controlled': 0.92,
 'dead': 0.9,
 'occur': 0.9,
 'disorder': 0.87,
 'program': 0.82,
 '##lat': 0.82,
 'through': 0.76,
 '##cl': 0.74,
 'computer': 0.71,
 '##ations': 0.7,
 'abbreviation': 0.69,
 'produced': 0.67,
 'is': 0.65,
 'center': 0.63,
 '"': 0.62,
 'produce': 0.62,
 'technology': 0.61,
 'process': 0.6,
 '##osing': 0.59,
 'matt': 0.54,
 'cc': 0.54,
 '##ation': 0.53,
 'outward': 0.53,
 'gage': 0.52,
 

In [364]:
def builder(records: list):
    ids = [x['id'] for x in records]
    contexts = [x['context'] for x in records]
    # create sparse vecs
    tokens = tokenizer(
        contexts, return_tensors='pt',
        padding=True, truncation=True
    )
    sparse_vecs = get_max_logits(model(**tokens), tokens)
    upserts = []
    for _id, sparse_vec, context in zip(ids, sparse_vecs, contexts):
        # extract columns where there are non-zero weights
        # indices = sparse_vec.squeeze().cpu().tolist()  # positions
        # values = sparse_vec[indices].cpu().tolist()  # weights/scores
        # build sparse values dictionary
        sparse_values = {
            "indices": indices,
            "values": values
        }
        # append all to upserts list as pinecone.Vector (or GRPCVector)
        upserts.append({
            'id': _id,
            'sparse_values': sparse_vec,
            'context': context
        })
    return upserts

In [365]:
builder(data[:3])[0]['sparse_values']

array([0., 0., 0., ..., 0., 0., 0.], dtype=float32)

In [19]:
display(data[:3])

pubmed[0]

[{'id': '21645374-0',
  'context': 'Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature.'},
 {'id': '21645374-1',
  'context': 'The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has been less studied during PCD in plants. The following paper elucidates the role of mitochondrial dynamics during developmentally regulated PCD in vivo in A.'},
 {'id': '21645374-2',
  '

{'pubid': 21645374,
 'question': 'Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?',
 'context': {'contexts': ['Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has been less studied during PCD in plants.',
   'The following paper elucidates the role of mitochondrial dynamics during developmentally regulated PCD in vivo in A. madagascariensis. A single areole within a window stage leaf (PCD is occurring) was divided into three areas based on the progression of PCD; cells that will not undergo PCD (NPCD), ce

In [ ]:
from milvus import default_server

from pymilvus import FieldSchema, CollectionSchema, DataType, Collection, utility, connections

In [7]:
default_server.start()

In [5]:
connections.connect(
    host = '127.0.0.1', 
    port = default_server.listen_port
)

print(utility.get_server_version())

v2.3.8-2-g99b8bbd82-lite


In [314]:
data[:1]
idk = builder(data[:2])
print(idk)
sparse_embeddings = [x['sparse_values'] for x in idk]
sparse_embeddings[0]

# number non zeros in sparse_embeddings[0]
len([x for x in sparse_embeddings[0] if x != 0])

[{'id': '21645374-0', 'sparse_values': [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 

174

In [366]:
upserts = builder(data[:100])

In [316]:
ids = [x['id'] for x in upserts]
sparse_embeddings = [x['sparse_values'] for x in upserts]
contexts = [x['context'] for x in upserts]

In [372]:
# find max lenght of an embedding value in embeddings
max_len = max([len(x) for x in sparse_embeddings])
print(max_len)

30522


In [157]:
# make all embeddings the same length
sparse_embeddings = [x + [0] * (max_len - len(x)) for x in sparse_embeddings]

In [320]:
assert all([len(x) == max_len for x in sparse_embeddings])

In [321]:
# Define the schema
fields = [
    FieldSchema(name="id", dtype=DataType.VARCHAR, max_length=36, is_primary=True),
    FieldSchema(name="embedding", dtype=DataType.FLOAT_VECTOR, dim=max_len), 
    FieldSchema(name="context", dtype=DataType.VARCHAR, max_length=65535)
]

schema = CollectionSchema(fields, "Schema for medical info")

# Create the collection if it doesn't exist
collection_name = "pubmed"
if not utility.has_collection(collection_name):
    collection = Collection(name=collection_name, schema=schema)
    print(f"Collection {collection_name} created.")
else:
    collection = Collection(name=collection_name)
    collection.drop()
    collection = Collection(name=collection_name, schema=schema)
    print(f"Collection {collection_name} already exists. but dropped and recreated.")


Collection pubmed already exists. but dropped and recreated.


In [322]:
print(collection.schema)
print(collection.name)

{'auto_id': False, 'description': 'Schema for medical info', 'fields': [{'name': 'id', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 36}, 'is_primary': True, 'auto_id': False}, {'name': 'embedding', 'description': '', 'type': <DataType.FLOAT_VECTOR: 101>, 'params': {'dim': 30522}}, {'name': 'context', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 65535}}], 'enable_dynamic_field': False}
pubmed


In [323]:
milvus_data = [
    ids,
    sparse_embeddings,
    contexts
]
print((milvus_data[1][0]))
len(milvus_data[1][0])

[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,

30522

In [324]:
assert all(len(x) == max_len for x in sparse_embeddings)

In [325]:
collection.insert(milvus_data)

(insert count: 100, delete count: 0, upsert count: 0, timestamp: 449942522023903235, success count: 100, err count: 0, cost: 0)

In [326]:
index_params = {
    "metric_type": "COSINE",
    "index_type": "IVF_FLAT",
    "params": {"nlist": 1024}
}

collection.create_index(field_name="embedding", index_params=index_params)

Status(code=0, message=)

In [327]:
collection.load()

In [329]:
data[:10]

[{'id': '21645374-0',
  'context': 'Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature.'},
 {'id': '21645374-1',
  'context': 'The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has been less studied during PCD in plants. The following paper elucidates the role of mitochondrial dynamics during developmentally regulated PCD in vivo in A.'},
 {'id': '21645374-2',
  '

In [378]:
# Example query embedding, replace with actual query
# query = "How were mitochondrial dynamics categorized in the study of programmed cell death (PCD) in A. madagascariensis, and what staining method was used to examine these dynamics?"
query = "How was visual acuity assessed in the study involving 100 patients and 13 healthy volunteers, and what criteria were used for correctly identifying the optotypes on the charts?"
# query = "What mechanism does the lace plant (Aponogeton madagascariensis) use to produce perforations in its leaves, and how does this process progress?"
# query = "Where is Madagascar located?"
query_chunked = chunker([query])
query_tokens = tokenizer(query_chunked[0], return_tensors="pt")
query_output = model(**query_tokens)

query_sparse_emb = get_max_logits(query_output, query_tokens)

search_params = {"metric_type": "COSINE", "params": {"nprobe": 10}}

results = collection.search(
    data=[query_sparse_emb],
    anns_field="embedding",
    param=search_params,
    limit=2,
    output_fields=["id", "context"],
)

for result in results[0]:
    print(f"Document ID: {result.id}, Context: {result.entity.get('context')}, Distance: {result.distance}")



Document ID: 16418930-2, Context: 100 patients (age 8 - 90 years, median 60.5 years) with various eye disorders, among them 39 with amblyopia due to strabismus, and 13 healthy volunteers were tested. Charts with the Snellen E and the Landolt C (Precision Vision) which mimic the ETDRS charts were used to assess visual acuity. Three out of 5 optotypes per line had to be correctly identified, while wrong answers were monitored., Distance: 0.5299680829048157
Document ID: 16418930-1, Context: Since optotypes are evaluated on individuals with good visual acuity and without eye disorders, differences in the lower visual acuity range cannot be excluded. In this study, visual acuity measured with the Snellen E was compared to the Landolt C acuity. 100 patients (age 8 - 90 years, median 60.5 years) with various eye disorders, among them 39 with amblyopia due to strabismus, and 13 healthy volunteers were tested., Distance: 0.48709961771965027


In [130]:
default_server.stop()

In [374]:
# get the first sparse embedding from the data
sparse_embeddings[0]

print(len(sparse_embeddings[0]))
print(len(query_sparse_emb))
print(len([x for x in sparse_embeddings[0] if x != 0]))
print(len([x for x in query_sparse_emb if x != 0]))

# cosine similarity between sparse_embeddings[0] and query_values
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

cosine_similarity([sparse_embeddings[0], query_sparse_emb])

30522
30522
174
69


array([[1.0000000e+00, 2.7299366e-04],
       [2.7299366e-04, 1.0000000e+00]])